In [6]:
# csv파일 두개를 결합하는 코드

import pandas as pd

# 1. 파일 불러오기 (인코딩 오류 시 encoding="utf-8-sig"로 변경)
crime_path = "경찰청_범죄 발생 지역별 통계_20251231.csv"
cctv_path = "CCTV정보.csv"

crime_df = pd.read_csv(crime_path, encoding="cp949")
cctv_df = pd.read_csv(cctv_path, encoding="cp949", low_memory=False)

# 2. 국내 지역 컬럼 추출 ('외국' 제외) 및 지역별 총 범죄발생건수 집계
domestic_regions = [
    col.strip() for col in crime_df.columns[2:]
    if not col.strip().startswith("외국")
]

crime_records = []
for region in domestic_regions:
    # 원본 컬럼명에 공백이 있을 수 있으므로 strip() 매칭
    matched_cols = [c for c in crime_df.columns[2:] if c.strip() == region]
    if matched_cols:
        total_crime = (
            pd.to_numeric(crime_df[matched_cols[0]], errors="coerce")
            .fillna(0)
            .sum()
        )
    else:
        total_crime = 0
    crime_records.append({
        "지역명": region,
        "총_범죄발생건수": int(round(total_crime))
    })

crime_summary_df = pd.DataFrame(crime_records)

# 3. CCTV 카메라 대수 숫자 변환 및 주소 기준 '시/도 시/군/구' 통일
cctv_df["카메라대수_num"] = pd.to_numeric(cctv_df["카메라대수"], errors="coerce").fillna(0)

sido_map = {
    "서울특별시": "서울", "서울": "서울",
    "부산광역시": "부산", "부산": "부산",
    "대구광역시": "대구", "대구": "대구",
    "인천광역시": "인천", "인천": "인천",
    "광주광역시": "광주", "광주": "광주",
    "대전광역시": "대전", "대전": "대전",
    "울산광역시": "울산", "울산": "울산",
    "세종특별자치시": "세종시", "세종시": "세종시",
    "경기도": "경기도", "경기": "경기도",
    "강원특별자치도": "강원도", "강원도": "강원도", "강원": "강원도",
    "충청북도": "충북", "충북": "충북",
    "충청남도": "충남", "충남": "충남",
    "전북특별자치도": "전북", "전라북도": "전북", "전북": "전북",
    "전라남도": "전남", "전남": "전남",
    "경상북도": "경북", "경북": "경북",
    "경상남도": "경남", "경남": "경남",
    "제주특별자치도": "제주", "제주도": "제주", "제주": "제주",
}
gwangju_gus = {"동구", "서구", "남구", "북구", "광산구"}

def make_region_name(row):
    addr = row.get("소재지도로명주소")
    if pd.isna(addr) or not str(addr).strip():
        addr = row.get("소재지지번주소")
    addr_str = str(addr or "").strip()
    if not addr_str:
        return ""

    toks = addr_str.split()
    t0 = toks[0].strip()
    t1 = toks[1].strip() if len(toks) > 1 else ""
    t2 = toks[2].strip() if len(toks) > 2 else ""

    if t0 == "산청군":
        return "경남 산청군"
    if t0 == "전남광주통합특별시":
        if t1 in gwangju_gus:
            return f"광주 {t1}"
        return f"전남 {t1}" if t1 else ""

    sido = sido_map.get(t0, "")
    if not sido:
        return ""
    if sido == "세종시" or t1 == "세종시":
        return "세종시"
    if not t1:
        return ""

    for g in ["중구", "동구", "서구", "남구", "북구"]:
        if t1.startswith(g) and t1 != g and not t1.endswith("동"):
            t1 = g
            break

    if t1.endswith("시") and t2.endswith("구"):
        return f"{sido} {t1}"
    return f"{sido} {t1}"

cctv_df["지역명"] = cctv_df.apply(make_region_name, axis=1)

cctv_summary_df = (
    cctv_df[cctv_df["지역명"].isin(domestic_regions)]
    .groupby("지역명", as_index=False)["카메라대수_num"]
    .sum()
)
cctv_summary_df["총_카메라대수"] = cctv_summary_df["카메라대수_num"].round().astype(int)
cctv_summary_df = cctv_summary_df[["지역명", "총_카메라대수"]]

# 4. 데이터 결합 및 가나다순 정렬
merged_df = pd.merge(crime_summary_df, cctv_summary_df, on="지역명", how="left")
merged_df["총_카메라대수"] = merged_df["총_카메라대수"].fillna(0).astype(int)
merged_df = merged_df.sort_values("지역명", ascending=True, kind="mergesort").reset_index(drop=True)

# 5. 문장 출력
for row in merged_df.itertuples(index=False):
    print(f"{row.지역명}에는 cctv가 {row.총_카메라대수:,}개고 범죄는 {row.총_범죄발생건수:,}개이다")

강원도 강릉시에는 cctv가 2,292개고 범죄는 5,924개이다
강원도 고성군에는 cctv가 672개고 범죄는 768개이다
강원도 동해시에는 cctv가 3,182개고 범죄는 2,132개이다
강원도 삼척시에는 cctv가 1,153개고 범죄는 1,727개이다
강원도 속초시에는 cctv가 0개고 범죄는 2,992개이다
강원도 양구군에는 cctv가 1,171개고 범죄는 500개이다
강원도 양양군에는 cctv가 767개고 범죄는 783개이다
강원도 영월군에는 cctv가 933개고 범죄는 1,048개이다
강원도 원주시에는 cctv가 3,992개고 범죄는 11,277개이다
강원도 인제군에는 cctv가 215개고 범죄는 757개이다
강원도 정선군에는 cctv가 1,204개고 범죄는 1,185개이다
강원도 철원군에는 cctv가 779개고 범죄는 1,140개이다
강원도 춘천시에는 cctv가 2,655개고 범죄는 8,131개이다
강원도 태백시에는 cctv가 15,951개고 범죄는 968개이다
강원도 평창군에는 cctv가 524개고 범죄는 1,102개이다
강원도 홍천군에는 cctv가 1,026개고 범죄는 1,938개이다
강원도 화천군에는 cctv가 0개고 범죄는 556개이다
강원도 횡성군에는 cctv가 774개고 범죄는 1,138개이다
경기도 가평군에는 cctv가 1,537개고 범죄는 2,155개이다
경기도 고양시에는 cctv가 9,625개고 범죄는 28,692개이다
경기도 과천시에는 cctv가 1,168개고 범죄는 1,769개이다
경기도 광명시에는 cctv가 1,028개고 범죄는 7,466개이다
경기도 광주시에는 cctv가 5,615개고 범죄는 11,929개이다
경기도 구리시에는 cctv가 2,669개고 범죄는 5,706개이다
경기도 군포시에는 cctv가 3,352개고 범죄는 6,434개이다
경기도 김포시에는 cctv가 6,694개고 범죄는 16,067개이다
경기도 남양주시에는 cctv가 6,344개고 범죄는 18,038개이다
경기도 동두천시에는 cctv가 2,794개고 범죄는